In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

processed = project_root / "data/processed"
lab09 = processed / "lab09"

columns = [
    "source_row", "timestamp", "amount_paid",
    "payment_currency", "payment_format",
]
validation = pd.read_parquet(
    processed / "lab02_validation.parquet", columns=columns
)
scores = np.load(lab09 / "with_sequence_validation_scores.npy")

# Each score must still point to the transaction that produced it.
np.testing.assert_array_equal(
    validation["source_row"], np.load(lab09 / "validation_source_rows.npy")
)
assert scores.shape == (len(validation),) and np.isfinite(scores).all()

ranked = validation.assign(
    day=validation["timestamp"].dt.normalize(),
    score=scores,
    tie_breaker=np.random.default_rng(42).random(len(validation)),
).sort_values(
    ["day", "score", "tie_breaker"], ascending=[True, False, True]
)

review_queue = ranked.groupby("day", sort=False).head(50)
alerts = (
    review_queue.sample(n=5, random_state=42)
    .sort_values("timestamp")[columns]
    .copy()
)

print(review_queue.groupby("day").size())
display(alerts)

day
2022-09-07    50
2022-09-08    50
dtype: int64


,source_row,timestamp,amount_paid,payment_currency,payment_format
232117,695464,2022-09-07 11:09:00,18533.55,US Dollar,ACH
304144,2273022,2022-09-07 14:50:00,9539.79,Euro,ACH
689404,3906793,2022-09-08 09:52:00,4428.75,Euro,ACH
824924,812704,2022-09-08 16:47:00,5960.62,Euro,ACH
964208,3863648,2022-09-08 23:56:00,6370.00,US Dollar,ACH


In [2]:
count_columns = ["sender_in_count_24h", "sender_out_count_24h"]

history = pd.read_parquet(
    processed / "lab03_history_features.parquet",
    columns=["source_row", *count_columns],
    filters=[("source_row", "in", alerts["source_row"].tolist())],
)

cases = alerts.merge(history, on="source_row", how="left", validate="one_to_one")
assert len(cases) == 5 and cases[count_columns].notna().all().all()


def describe_alert(row) -> str:
    return (
        f"The account sent an {row.payment_format} payment. "
        f"The amount was {row.amount_paid:,.2f} "
        f"(currency: {row.payment_currency}). "
        f"In the preceding 24 hours, the account received "
        f"{row.sender_in_count_24h} incoming and sent "
        f"{row.sender_out_count_24h} outgoing payments."
    )


cases["query_text"] = [describe_alert(row) for row in cases.itertuples()]

cache_dir = processed / "lab12"
cache_dir.mkdir(parents=True, exist_ok=True)
cases.to_parquet(cache_dir / "alert_queries.parquet", index=False)

for row in cases.itertuples():
    print(f"{row.source_row} | {row.query_text}\n")

695464 | The account sent an ACH payment. The amount was 18,533.55 (currency: US Dollar). In the preceding 24 hours, the account received 3 incoming and sent 2 outgoing payments.

2273022 | The account sent an ACH payment. The amount was 9,539.79 (currency: Euro). In the preceding 24 hours, the account received 1 incoming and sent 2 outgoing payments.

3906793 | The account sent an ACH payment. The amount was 4,428.75 (currency: Euro). In the preceding 24 hours, the account received 2 incoming and sent 4 outgoing payments.

812704 | The account sent an ACH payment. The amount was 5,960.62 (currency: Euro). In the preceding 24 hours, the account received 5 incoming and sent 4 outgoing payments.

3863648 | The account sent an ACH payment. The amount was 6,370.00 (currency: US Dollar). In the preceding 24 hours, the account received 6 incoming and sent 10 outgoing payments.



In [3]:
cases

,source_row,timestamp,amount_paid,payment_currency,payment_format,sender_in_count_24h,sender_out_count_24h,query_text
0,695464,2022-09-07 11:09:00,18533.55,US Dollar,ACH,3,2,The account sent an ACH payment. The amount wa...
1,2273022,2022-09-07 14:50:00,9539.79,Euro,ACH,1,2,The account sent an ACH payment. The amount wa...
2,3906793,2022-09-08 09:52:00,4428.75,Euro,ACH,2,4,The account sent an ACH payment. The amount wa...
3,812704,2022-09-08 16:47:00,5960.62,Euro,ACH,5,4,The account sent an ACH payment. The amount wa...
4,3863648,2022-09-08 23:56:00,6370.00,US Dollar,ACH,6,10,The account sent an ACH payment. The amount wa...


In [7]:
source_url = (
    "https://fintrac-canafe.canada.ca/guidance-directives/"
    "transaction-operation/indicators-indicateurs/fin_mltf-eng"
)

passages = pd.DataFrame(
    [
        ("P01", "Client profile", "Payment volume differs from the customer's usual activity."),
        ("P02", "Products and services", "A customer overpays a credit card and then requests a cash advance."),
        ("P03", "Atypical activity", "Funds enter and leave the same account over a short interval."),
        ("P04", "Identification", "Identity records supplied by a customer contain inconsistent details."),
        ("P05", "Changes in account activity", "Payments resume after an account has been dormant."),
        ("P06", "Client profile", "The size or type of a payment differs from what is expected for the customer."),
        ("P07", "Products and services", "An account receives money and passes it onward to beneficiaries."),
        ("P08", "Client behaviour", "The customer gives an unclear explanation of a transaction's purpose."),
        ("P09", "Atypical activity", "Transactions repeatedly occur at a particular time of day."),
        ("P10", "Products and services", "Activity suggests a personal account is being used for business."),
        ("P11", "Changes in account activity", "Business account activity increases after a change of ownership."),
        ("P12", "Atypical activity", "A customer frequently exchanges smaller banknotes for larger denominations."),
    ],
    columns=["passage_id", "source_section", "text"],
)
passages["source_url"] = source_url
passages.to_parquet(cache_dir / "guidance_passages.parquet", index=False)

with pd.option_context("display.max_colwidth", None):
    display(passages[["passage_id", "text"]])
    display(cases[["source_row", "query_text"]])

,passage_id,text
0,P01,Payment volume differs from the customer's usual activity.
1,P02,A customer overpays a credit card and then requests a cash advance.
2,P03,Funds enter and leave the same account over a short interval.
3,P04,Identity records supplied by a customer contain inconsistent details.
4,P05,Payments resume after an account has been dormant.
5,P06,The size or type of a payment differs from what is expected for the customer.
6,P07,An account receives money and passes it onward to beneficiaries.
7,P08,The customer gives an unclear explanation of a transaction's purpose.
8,P09,Transactions repeatedly occur at a particular time of day.
9,P10,Activity suggests a personal account is being used for business.


,source_row,query_text
0,695464,"The account sent an ACH payment. The amount was 18,533.55 (currency: US Dollar). In the preceding 24 hours, the account received 3 incoming and sent 2 outgoing payments."
1,2273022,"The account sent an ACH payment. The amount was 9,539.79 (currency: Euro). In the preceding 24 hours, the account received 1 incoming and sent 2 outgoing payments."
2,3906793,"The account sent an ACH payment. The amount was 4,428.75 (currency: Euro). In the preceding 24 hours, the account received 2 incoming and sent 4 outgoing payments."
3,812704,"The account sent an ACH payment. The amount was 5,960.62 (currency: Euro). In the preceding 24 hours, the account received 5 incoming and sent 4 outgoing payments."
4,3863648,"The account sent an ACH payment. The amount was 6,370.00 (currency: US Dollar). In the preceding 24 hours, the account received 6 incoming and sent 10 outgoing payments."


In [9]:
relevant = {
    695464:  ["P03"],
    2273022: ["P03"],
    3906793: ["P03"],
    812704:  ["P03"],
    3863648: ["P03"],
}

assert set(relevant) == set(cases["source_row"])
valid_ids = set(passages["passage_id"])

for source_row, selected in relevant.items():
    assert isinstance(selected, list), f"Finish reviewing alert {source_row}"
    assert set(selected) <= valid_ids, f"Unknown passage ID for {source_row}"
    assert len(selected) == len(set(selected)), f"Repeated ID for {source_row}"

judgments = pd.DataFrame([
    {
        "source_row": source_row,
        "passage_id": passage_id,
        "relevant": passage_id in selected,
    }
    for source_row, selected in relevant.items()
    for passage_id in passages["passage_id"]
])

judgments.to_parquet(cache_dir / "relevance_judgments.parquet", index=False)
print(f"Saved {len(judgments)} judgments.")
display(judgments.groupby("source_row")["relevant"].sum().rename("selected_count"))

Saved 60 judgments.


source_row
695464     1
812704     1
2273022    1
3863648    1
3906793    1
Name: selected_count, dtype: int64

In [11]:
judgments

,source_row,passage_id,relevant
0,695464,P01,False
1,695464,P02,False
2,695464,P03,True
3,695464,P04,False
4,695464,P05,False
5,695464,P06,False
6,695464,P07,False
7,695464,P08,False
8,695464,P09,False
9,695464,P10,False


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch

# TF-IDF learns its vocabulary from the 12 searchable passages.
tfidf = TfidfVectorizer(ngram_range=(1, 2), dtype=np.float32)
passage_terms = tfidf.fit_transform(passages["text"])
alert_terms = tfidf.transform(cases["query_text"])
tfidf_scores = cosine_similarity(alert_terms, passage_terms)

# Reuse the original encoder saved in Lab 11.
torch.set_num_threads(2)
encoder = SentenceTransformer(
    str(project_root / "models/lab11_original_encoder"),
    device="cpu",
    local_files_only=True,
)

passage_vectors = encoder.encode(
    passages["text"].tolist(), normalize_embeddings=True
)
alert_vectors = encoder.encode(
    cases["query_text"].tolist(), normalize_embeddings=True
)
embedding_scores = alert_vectors @ passage_vectors.T

assert tfidf_scores.shape == embedding_scores.shape == (5, 12)

/Users/sara/Projects/data-science-lab/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 16:28:28.963000 74766 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6765.32it/s]


In [12]:
relevant_ids = judgments.loc[judgments["relevant"]].set_index("source_row")["passage_id"]
assert relevant_ids.index.is_unique  # One selected passage per alert in this exercise.

passage_ids = passages["passage_id"].to_numpy()
records = []

for method, scores in [
    ("TF-IDF", tfidf_scores),
    ("Original encoder", embedding_scores),
]:
    for source_row, score_row in zip(cases["source_row"], scores, strict=True):
        order = np.argsort(-score_row, kind="stable")
        ranked_ids = passage_ids[order].tolist()
        selected_id = relevant_ids.loc[source_row]

        records.append({
            "source_row": source_row,
            "method": method,
            "selected_passage": selected_id,
            "top_three": ", ".join(ranked_ids[:3]),
            "selected_rank": ranked_ids.index(selected_id) + 1,
        })

comparison = pd.DataFrame(records)
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)
comparison.to_csv(reports_dir / "lab_12_retrieval_comparison.csv", index=False)
display(comparison)

,source_row,method,selected_passage,top_three,selected_rank
0,695464,TF-IDF,P03,"P06, P03, P01",2
1,2273022,TF-IDF,P03,"P06, P03, P01",2
2,3906793,TF-IDF,P03,"P06, P03, P01",2
3,812704,TF-IDF,P03,"P06, P03, P01",2
4,3863648,TF-IDF,P03,"P06, P03, P01",2
5,695464,Original encoder,P03,"P07, P03, P05",2
6,2273022,Original encoder,P03,"P07, P03, P05",2
7,3906793,Original encoder,P03,"P07, P03, P05",2
8,812704,Original encoder,P03,"P07, P03, P05",2
9,3863648,Original encoder,P03,"P07, P03, P05",2
